# Tutorial 04: Análisis de Onda PPG y Entrenamiento Consciente de Cuantización (QAT)
### TinyML para Raspberry Pi Pico 2 / Pico 2 W (RP2350 - Cortex-M33) con TensorFlow QAT

En este tutorial aprenderás:
1. **Generación de señales de Fotopletismografía (PPG)** con `neurokit2`: Pulso Limpio, Pulso Ectópico/Arrítmico y Artefacto de Movimiento Severo.
2. **Diferencias críticas entre PTQ y QAT**:
   - **PTQ (Post-Training Quantization)**: Cuantiza una red ya entrenada en FP32. Puede sufrir pérdida de precisión cuando las características tienen distribuciones asimétricas o rangos dinámicos amplios.
   - **QAT (Quantization-Aware Training)**: Inserta nodos de cuantización simulada (*fake quantization*) durante el pase hacia adelante (*forward pass*) y calcula gradientes utilizando el estimador directo (*Straight-Through Estimator, STE*). La red aprende a ser robusta a la pérdida de precisión de 8 bits durante el propio entrenamiento.
3. **Conversión a FlatBuffer TFLite INT8** sin degradación de exactitud.
4. **Despliegue y benchmarking en Raspberry Pi Pico 2 (RP2350)** con Arduino IDE.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.signal_generator import generate_ppg_dataset
from utils.exporter import tflite_to_c_header, save_test_samples_header
from utils.tflite_eval import evaluate_tflite_model

## 1. Generación de Señales PPG con NeuroKit2

In [ ]:
SAMPLE_LEN = 150
SAMPLING_RATE = 100
CLASS_NAMES = ["Normal", "Ectopic", "Motion_Artifact"]

X, y = generate_ppg_dataset(num_samples_per_class=400, sample_len=SAMPLE_LEN, sampling_rate=SAMPLING_RATE)
X = np.expand_dims(X, axis=-1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for c in range(3):
    idx = np.where(y_train == c)[0][0]
    axes[c].plot(X_train[idx, :, 0], color=["crimson", "navy", "orange"][c])
    axes[c].set_title(f"PPG: {CLASS_NAMES[c]}", fontweight='bold')
    axes[c].grid(True, alpha=0.5)
axes[0].set_ylabel("Amplitud Normalizada")
plt.tight_layout()
plt.show()

## 2. Modelo Base y Entrenamiento Previo en Float32

In [ ]:
def build_ppg_model():
    inputs = tf.keras.Input(shape=(SAMPLE_LEN, 1))
    x = tf.keras.layers.Conv1D(16, 7, strides=2, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPool1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 5, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(24, activation="relu")(x)
    outputs = tf.keras.layers.Dense(3, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

base_model = build_ppg_model()
base_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
base_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=15, batch_size=32, verbose=1)

_, base_acc = base_model.evaluate(X_test, y_test, verbose=0)
print(f"Exactitud Modelo Base FP32: {base_acc * 100:.2f}%")

## 3. Entrenamiento Consciente de Cuantización (QAT) con TFMOT

In [ ]:
import tensorflow_model_optimization as tfmot

qat_model = tfmot.quantization.keras.quantize_model(base_model)
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
qat_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=8, batch_size=32, verbose=1)

_, qat_acc = qat_model.evaluate(X_test, y_test, verbose=0)
print(f"Exactitud tras QAT Fine-Tuning: {qat_acc * 100:.2f}%")

## 4. Conversión a TFLite INT8 y Exportación de Encabezados C (`model_data.h`)

In [ ]:
def rep_dataset():
    for i in range(150):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_qat_int8 = converter.convert()

ARDUINO_DIR = Path("../arduino/ppg_inference_pico2")
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)

tflite_to_c_header(tflite_qat_int8, str(ARDUINO_DIR / "model_data.h"))
save_test_samples_header(X_test[:, :, 0], y_test, str(ARDUINO_DIR / "test_samples.h"), CLASS_NAMES, samples_per_class=3)
print("Archivos C de PPG exportados listos para Arduino IDE!")